## Contexte et Problématique 

1- Contexte du problème

L’agriculture en Afrique de l’Ouest est fortement influencée par des facteurs climatiques, pédologiques et socio-économiques. L’objectif de ce projet est de construire un modèle capable de prédire le rendement agricole à partir de données multi-sources (FAOSTAT, météo et sol).

2- Problème ML

Il s’agit d’un problème de régression visant à prédire le rendement agricole (yield) en fonction des conditions environnementales et agricoles.

3- Variables utilisées

Variables explicatives :
température moyenne
précipitations
pH du sol
matière organique
type de culture
Variable cible :
rendement agricole (yield)

# Données

#### Chargement des données 

In [16]:
import pandas as pd

faostat_df = pd.read_csv("../data/raw/faostat.csv")
meteo_df = pd.read_csv("../data/raw/meteo.csv")
soil_df = pd.read_csv("../data/raw/soil.csv")

#### Vérifier la structure 

In [17]:
faostat_df.head()

,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
0,QCL,Crops and livestock products,204,Benin,5312,Area harvested,1520.01,"Cassava, fresh",2000,2000,ha,219404.0,A,Official figure,NaN
1,QCL,Crops and livestock products,204,Benin,5412,Yield,1520.01,"Cassava, fresh",2000,2000,kg/ha,10711.8,A,Official figure,NaN
2,QCL,Crops and livestock products,204,Benin,5510,Production,1520.01,"Cassava, fresh",2000,2000,t,2350210.0,A,Official figure,NaN
3,QCL,Crops and livestock products,204,Benin,5312,Area harvested,1520.01,"Cassava, fresh",2001,2001,ha,240048.0,A,Official figure,NaN
4,QCL,Crops and livestock products,204,Benin,5412,Yield,1520.01,"Cassava, fresh",2001,2001,kg/ha,11262.2,A,Official figure,NaN


In [18]:
meteo_df.head()

,country,year,avg_temp_c,total_precip_mm
0,Benin,2000,26.907377,1298.6
1,Benin,2001,26.925205,984.6
2,Benin,2002,27.001096,1332.9
3,Benin,2003,27.223288,1153.4
4,Benin,2004,26.996721,983.2


In [19]:
soil_df.head()

,country,soil_ph,organic_matter,clay_pct
0,Benin,6.2,242,167
1,Senegal,6.6,77,195
2,Ghana,6.2,247,273
3,Nigeria,5.8,294,211
4,Mali,7.8,69,310


#### Harmoniser les données pour les fusionner

In [20]:
faostat_df = faostat_df.rename(columns={
    "Area": "country",
    "Year": "year"
})

In [21]:
# Nettoyage des espaces
faostat_df["country"] = faostat_df["country"].str.strip()
meteo_df["country"] = meteo_df["country"].str.strip()
soil_df["country"] = soil_df["country"].str.strip()

#### Jointure

In [22]:
# faostat + météo 

df_1 = faostat_df.merge(
    meteo_df,
    on=["country", "year"],
    how="left"
)


In [23]:
# + sol
df= df_1.merge(
    soil_df,
    on="country",
    how="left"
)


In [31]:
df.to_csv("../data/processed/dataset_final.csv", index=False)

--- 
# Analyse Exploratoire des données

### Détection des valeurs manquantes

In [25]:
somme = df.isnull().sum()

frequence = df.isnull().mean() * 100
print("Somme des valeurs manquantes :")

tableau = pd.DataFrame({
    "Somme des valeurs manquantes": somme,
    "Fréquence des valeurs manquantes (%)": frequence
})
print(tableau)

Somme des valeurs manquantes :
                  Somme des valeurs manquantes  \
Domain Code                                  0   
Domain                                       0   
Area Code (M49)                              0   
country                                      0   
Element Code                                 0   
Element                                      0   
Item Code (CPC)                              0   
Item                                         0   
Year Code                                    0   
year                                         0   
Unit                                         0   
Value                                        0   
Flag                                         0   
Flag Description                             0   
Note                                      2474   
avg_temp_c                                 360   
total_precip_mm                            360   
soil_ph                                    360   
organic_matter     

### Détection des outliers 

In [32]:
import pandas as pd
from sklearn.ensemble import IsolationForest


# Garder uniquement les colonnes numériques
X = df.select_dtypes(include=["number"]).dropna()

# Modèle Isolation Forest
iso = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42
)

# Détection des outliers
pred = iso.fit_predict(X)

# Ajouter les résultats au DataFrame
X = X.copy()
X["outlier"] = pred
X["outlier_flag"] = X["outlier"].map({1: 0, -1: 1})

# Récupérer les outliers
outliers = X[X["outlier_flag"] == 1]

print("Nombre d'outliers détectés :", len(outliers))
outliers.head()

Nombre d'outliers détectés : 108


,Area Code (M49),Element Code,Item Code (CPC),Year Code,year,Value,avg_temp_c,total_precip_mm,soil_ph,organic_matter,clay_pct,outlier,outlier_flag
2,204,5510,1520.01,2000,2000,2350210.00,26.907377,1298.6,6.2,242.0,167.0,-1,1
68,204,5510,1520.01,2022,2022,4350053.57,26.126027,978.3,6.2,242.0,167.0,-1,1
71,204,5510,1520.01,2023,2023,4449430.00,26.768219,857.6,6.2,242.0,167.0,-1,1
362,854,5510,1520.01,2000,2000,4433.19,27.984153,612.5,6.3,86.0,163.0,-1,1
429,854,5312,1520.01,2023,2023,1072.00,28.155890,699.6,6.3,86.0,163.0,-1,1


### Détection des doublons

In [ ]:
doublons = df.duplicated().sum()

print("Nombre de doublons :", doublons)



Nombre de doublons : 0


---
# Modélisation Machine Learning

## 2. Chargement du dataset final

On charge directement le dataset unifié issu de la jointure FAOSTAT + Météo + Sol, **sans aucun nettoyage** des données.

In [26]:
import pandas as pd

df = pd.read_csv("../data/processed/dataset_final.csv")

print(f"Shape : {df.shape}")
print(f"Colonnes : {list(df.columns)}")
df.head()

Shape : (2520, 20)
Colonnes : ['Domain Code', 'Domain', 'Area Code (M49)', 'country', 'Element Code', 'Element', 'Item Code (CPC)', 'Item', 'Year Code', 'year', 'Unit', 'Value', 'Flag', 'Flag Description', 'Note', 'avg_temp_c', 'total_precip_mm', 'soil_ph', 'organic_matter', 'clay_pct']


,Domain Code,Domain,Area Code (M49),country,Element Code,Element,Item Code (CPC),Item,Year Code,year,Unit,Value,Flag,Flag Description,Note,avg_temp_c,total_precip_mm,soil_ph,organic_matter,clay_pct
0,QCL,Crops and livestock products,204,Benin,5312,Area harvested,1520.01,"Cassava, fresh",2000,2000,ha,219404.0,A,Official figure,NaN,26.907377,1298.6,6.2,242.0,167.0
1,QCL,Crops and livestock products,204,Benin,5412,Yield,1520.01,"Cassava, fresh",2000,2000,kg/ha,10711.8,A,Official figure,NaN,26.907377,1298.6,6.2,242.0,167.0
2,QCL,Crops and livestock products,204,Benin,5510,Production,1520.01,"Cassava, fresh",2000,2000,t,2350210.0,A,Official figure,NaN,26.907377,1298.6,6.2,242.0,167.0
3,QCL,Crops and livestock products,204,Benin,5312,Area harvested,1520.01,"Cassava, fresh",2001,2001,ha,240048.0,A,Official figure,NaN,26.925205,984.6,6.2,242.0,167.0
4,QCL,Crops and livestock products,204,Benin,5412,Yield,1520.01,"Cassava, fresh",2001,2001,kg/ha,11262.2,A,Official figure,NaN,26.925205,984.6,6.2,242.0,167.0


## 3. Préparation des features et de la cible

On filtre les lignes **`Element == 'Yield'`** (rendement agricole), notre variable cible.

**Features retenues :**
- `avg_temp_c` — température moyenne (°C)
- `total_precip_mm` — précipitations totales (mm)
- `soil_ph` — pH du sol
- `organic_matter` — matière organique
- `clay_pct` — taux d'argile (%)
- `Item` — type de culture (encodé numériquement)
- `year` — année

**Variable cible :** `Value` (rendement agricole)

>  **Aucun nettoyage effectué.** Les valeurs manquantes (NaN) sont gérées directement dans les pipelines des modèles via un `SimpleImputer` — l'imputation fait partie du modèle, pas du preprocessing des données.

In [27]:
# Filtrer uniquement les lignes 'Yield'
df_yield = df[df["Element"] == "Yield"].copy()
print(f"Lignes 'Yield' retenues : {len(df_yield)}")

# Définition des features et de la cible
features = [
    "avg_temp_c",
    "total_precip_mm",
    "soil_ph",
    "organic_matter",
    "clay_pct",
    "Item",
    "year"
]

X = df_yield[features]
y = df_yield["Value"]

# Encodage de la variable catégorielle 'Item' avec get_dummies
X = pd.get_dummies(X, columns=["Item"], drop_first=True)

# Vérifications
print(f"\nShape X : {X.shape} | Shape y : {y.shape}")

print("\nColonnes après encodage :")
print(X.columns)

print("\nValeurs manquantes dans X :")
print(X.isnull().sum())

Lignes 'Yield' retenues : 840

Shape X : (840, 10) | Shape y : (840,)

Colonnes après encodage :
Index(['avg_temp_c', 'total_precip_mm', 'soil_ph', 'organic_matter',
       'clay_pct', 'year', 'Item_Maize (corn)', 'Item_Millet', 'Item_Rice',
       'Item_Sorghum'],
      dtype='str')

Valeurs manquantes dans X :
avg_temp_c           120
total_precip_mm      120
soil_ph              120
organic_matter       120
clay_pct             120
year                   0
Item_Maize (corn)      0
Item_Millet            0
Item_Rice              0
Item_Sorghum           0
dtype: int64


## 4. Split Train / Test

Découpage **80% / 20%** avec `random_state=42` pour la reproductibilité.

In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Jeu d'entraînement : {X_train.shape[0]} lignes")
print(f"Jeu de test        : {X_test.shape[0]} lignes")

Jeu d'entraînement : 672 lignes
Jeu de test        : 168 lignes


## 5. Entraînement des modèles

Chaque modèle est enveloppé dans un **Pipeline** `SimpleImputer → Modèle` afin de traiter les NaN sans toucher aux données brutes. Les hyperparamètres sont ceux par défaut.

Métriques d'évaluation :
- **MAE** — Mean Absolute Error
- **RMSE** — Root Mean Squared Error
- **R²** — Coefficient de détermination

In [29]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


model_lr = LinearRegression()

model_dt = DecisionTreeRegressor(
    random_state=42
)

model_rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

### 5.1 Régression Linéaire (Linear Regression)

In [30]:
# Entraînement du modèle
model_lr.fit(X_train, y_train)

# Prédictions
y_pred_lr = model_lr.predict(X_test)

# Évaluation
mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr   = r2_score(y_test, y_pred_lr)

# Résultats
print("=== Linear Regression ===")
print(f"MAE  : {mae_lr:,.2f}")
print(f"RMSE : {rmse_lr:,.2f}")
print(f"R²   : {r2_lr:.4f}")

ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

### 5.2 Arbre de Décision (Decision Tree Regressor)

In [ ]:
# Entraînement
model_dt.fit(X_train, y_train)

# Prédictions
y_pred_dt = model_dt.predict(X_test)

# Évaluation
mae_dt  = mean_absolute_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))
r2_dt   = r2_score(y_test, y_pred_dt)

# Résultats
print("\n=== Decision Tree Regressor ===")
print(f"MAE  : {mae_dt:,.2f}")
print(f"RMSE : {rmse_dt:,.2f}")
print(f"R²   : {r2_dt:.4f}")

### 5.3 Forêt Aléatoire (Random Forest Regressor)

In [ ]:
# Entraînement
model_rf.fit(X_train, y_train)

# Prédictions
y_pred_rf = model_rf.predict(X_test)

# Évaluation
mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf   = r2_score(y_test, y_pred_rf)

# Résultats
print("\n=== Random Forest Regressor ===")
print(f"MAE  : {mae_rf:,.2f}")
print(f"RMSE : {rmse_rf:,.2f}")
print(f"R²   : {r2_rf:.4f}")

## 6. Comparaison des performances

Tableau récapitulatif trié par **R²** décroissant.

In [ ]:
results = pd.DataFrame({
    "Modèle": ["Linear Regression", "Decision Tree", "Random Forest"],
    "MAE":    [mae_lr,  mae_dt,  mae_rf],
    "RMSE":   [rmse_lr, rmse_dt, rmse_rf],
    "R²":     [r2_lr,   r2_dt,   r2_rf]
})

results = results.sort_values("R²", ascending=False).reset_index(drop=True)
results.index += 1
results["MAE"]  = results["MAE"].map("{:,.2f}".format)
results["RMSE"] = results["RMSE"].map("{:,.2f}".format)
results["R²"]   = results["R²"].map("{:.4f}".format)

display(results)

## Nettoyage
Le jeu de données initial extrait de la FAOSTAT concerne la production agricole en afrique de l'oeust. Dans sa version brute, le fichier utilise une structure dite "longue" (Tidy Data).Pour une même année, un même pays et un même produit, les données sont séparées sur 3 lignes distinctes car elles mélangent trois variables (Element) dans la même colonne Value : La superficie récoltée (ha),la superficie récoltée (ha),la production totale (t).

Nous avons donc isoler la superficie, le rendement et la production dans des colonnes distinctes et retirer les colonnes et retirer les colonnes inutiles du dataset: Domain Code, Domain, Area Code (M49), Element Code, Item, Year Code, Unit, Flag, Flag Description, Note

In [ ]:
df=pd.read_csv("../data/processed/dataset_final.csv")
print(f"Shape : {df.shape}")
print(f"Colonnes : {list(df.columns)}")
df.head()

In [ ]:
'''on effectue le pivot en supposant que les colonnes clés sont 'country', 'Item', 'year', 'avg_temp_c', 'total_precip_mm', 'soil_ph', 'organic_matter' et 'clay_pct' 
puis on renomme les colonnes pour inclure les unités explicitement.On s'assure egalemnt de retirer les colonnes inutiles pour le pivot'''


df_pivot = df.pivot(
    index=['country', 'Item', 'year',"avg_temp_c", "total_precip_mm", "soil_ph", "organic_matter", "clay_pct"], 
    columns='Element',               
    values='Value'                   
).reset_index()

df_pivot = df_pivot.rename(columns={
    'Area harvested': 'Superficie (ha)',
    'Yield': 'Rendement (kg/ha)',
    'Production': 'Production (t)'
})

print(df_pivot.head())

In [ ]:
df_pivot.to_csv("../data/processed/dataset_clean.csv", index=False)

---
# Modélisation Machine Learning sur le Dataset Nettoyé

## 7. Chargement du dataset nettoyé

On charge le dataset issu du nettoyage (pivot), où chaque ligne correspond à une observation unique (pays, culture, année).

In [ ]:
import pandas as pd

df_clean = pd.read_csv("../data/processed/dataset_clean.csv")

print(f"Shape : {df_clean.shape}")
print(f"Colonnes : {list(df_clean.columns)}")
df_clean.head()

## 8. Préparation des features et de la cible

**Features retenues :**
- `avg_temp_c` — température moyenne (°C)
- `total_precip_mm` — précipitations totales (mm)
- `soil_ph` — pH du sol
- `organic_matter` — matière organique
- `clay_pct` — taux d'argile (%)
- `Item` — type de culture (encodé numériquement)
- `year` — année

**Variable cible :** `Rendement (kg/ha)`

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Suppression des lignes où la cible est manquante
df_clean_model = df_clean.dropna(subset=["Rendement (kg/ha)"]).copy()
print(f"Lignes retenues : {len(df_clean_model)}")

# Encodage de la variable catégorielle 'Item' (type de culture)
le_cl = LabelEncoder()
df_clean_model["Item_encoded"] = le_cl.fit_transform(df_clean_model["Item"].astype(str))
print(f"Cultures encodées : {dict(zip(le_cl.classes_, le_cl.transform(le_cl.classes_)))}")

# Définition des features (X) et de la cible (y)
features_cl = ["avg_temp_c", "total_precip_mm", "soil_ph", "organic_matter", "clay_pct", "Item_encoded", "year"]
X_cl = df_clean_model[features_cl]
y_cl = df_clean_model["Rendement (kg/ha)"]

print(f"\nShape X : {X_cl.shape} | Shape y : {y_cl.shape}")
print(f"Valeurs manquantes dans X :\n{X_cl.isnull().sum()}")

## 9. Split Train / Test

Découpage **80% / 20%** avec `random_state=42` pour la reproductibilité.

In [ ]:
from sklearn.model_selection import train_test_split

X_train_cl, X_test_cl, y_train_cl, y_test_cl = train_test_split(
    X_cl, y_cl, test_size=0.2, random_state=42
)

print(f"Jeu d'entraînement : {X_train_cl.shape[0]} lignes")
print(f"Jeu de test        : {X_test_cl.shape[0]} lignes")

## 10. Entraînement des modèles

Même structure de pipelines `SimpleImputer → Modèle` que précédemment.

Métriques d'évaluation :
- **MAE** — Mean Absolute Error
- **RMSE** — Root Mean Squared Error
- **R²** — Coefficient de détermination

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Définition des pipelines
pipe_lr_cl = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model",   LinearRegression())
])

pipe_dt_cl = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model",   DecisionTreeRegressor(random_state=42))
])

pipe_rf_cl = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model",   RandomForestRegressor(n_estimators=100, random_state=42))
])

### 10.1 Régression Linéaire (Linear Regression)

In [ ]:
pipe_lr_cl.fit(X_train_cl, y_train_cl)
y_pred_lr_cl = pipe_lr_cl.predict(X_test_cl)

mae_lr_cl  = mean_absolute_error(y_test_cl, y_pred_lr_cl)
rmse_lr_cl = np.sqrt(mean_squared_error(y_test_cl, y_pred_lr_cl))
r2_lr_cl   = r2_score(y_test_cl, y_pred_lr_cl)

print("=== Linear Regression ===")
print(f"  MAE  : {mae_lr_cl:,.2f}")
print(f"  RMSE : {rmse_lr_cl:,.2f}")
print(f"  R²   : {r2_lr_cl:.4f}")

### 10.2 Arbre de Décision (Decision Tree Regressor)

In [ ]:
pipe_dt_cl.fit(X_train_cl, y_train_cl)
y_pred_dt_cl = pipe_dt_cl.predict(X_test_cl)

mae_dt_cl  = mean_absolute_error(y_test_cl, y_pred_dt_cl)
rmse_dt_cl = np.sqrt(mean_squared_error(y_test_cl, y_pred_dt_cl))
r2_dt_cl   = r2_score(y_test_cl, y_pred_dt_cl)

print("=== Decision Tree Regressor ===")
print(f"  MAE  : {mae_dt_cl:,.2f}")
print(f"  RMSE : {rmse_dt_cl:,.2f}")
print(f"  R²   : {r2_dt_cl:.4f}")

### 10.3 Forêt Aléatoire (Random Forest Regressor)

In [ ]:
pipe_rf_cl.fit(X_train_cl, y_train_cl)
y_pred_rf_cl = pipe_rf_cl.predict(X_test_cl)

mae_rf_cl  = mean_absolute_error(y_test_cl, y_pred_rf_cl)
rmse_rf_cl = np.sqrt(mean_squared_error(y_test_cl, y_pred_rf_cl))
r2_rf_cl   = r2_score(y_test_cl, y_pred_rf_cl)

print("=== Random Forest Regressor ===")
print(f"  MAE  : {mae_rf_cl:,.2f}")
print(f"  RMSE : {rmse_rf_cl:,.2f}")
print(f"  R²   : {r2_rf_cl:.4f}")

## 11. Comparaison des performances

Tableau récapitulatif trié par **R²** décroissant.

In [ ]:
results_cl = pd.DataFrame({
    "Modèle": ["Linear Regression", "Decision Tree", "Random Forest"],
    "MAE":    [mae_lr_cl,  mae_dt_cl,  mae_rf_cl],
    "RMSE":   [rmse_lr_cl, rmse_dt_cl, rmse_rf_cl],
    "R²":     [r2_lr_cl,   r2_dt_cl,   r2_rf_cl]
})

results_cl = results_cl.sort_values("R²", ascending=False).reset_index(drop=True)
results_cl.index += 1
results_cl["MAE"]  = results_cl["MAE"].map("{:,.2f}".format)
results_cl["RMSE"] = results_cl["RMSE"].map("{:,.2f}".format)
results_cl["R²"]   = results_cl["R²"].map("{:.4f}".format)

display(results_cl)